# 03 - Phase 4 Evaluation

Honest classifier metrics from manually inspected predictions.

This notebook uses 100 proportional records plus 25 extra predicted-Goods records. The proportional subset protects the headline accuracy comparison; the extra Goods rows make minority-class behavior easier to inspect.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "notebooks"))

import phase4_eval as p4

print(f"Project root: {project_root}")
print(f"Enriched data: {p4.DATA_PATH} (exists={p4.DATA_PATH.exists()})")
print(f"Final annotations: {p4.FINAL_ANNOTATIONS} (exists={p4.FINAL_ANNOTATIONS.exists()})")

## Task 1 - Stratified Sample

Sampling plan: 54 Works, 38 Services, 8 Goods, plus 25 extra Goods. The script keeps a `sample_group` marker so the notebook can distinguish headline proportional metrics from diagnostic all-row metrics.

In [ ]:
records = p4.load_data()
sample_rows = p4.stratified_sample(records)

p4.print_distribution(records)
print("\nSample rows:", len(sample_rows))
print("Proportional rows:", sum(row["sample_group"] == "proportional_100" for row in sample_rows))
print("Goods oversample rows:", sum(row["sample_group"] == "goods_oversample" for row in sample_rows))

p4.write_annotation_template(sample_rows)
print(f"Annotation template ready: {p4.TEMPLATE_PATH}")

## Task 2 - Manual Annotation

The final CSV should contain one row per sampled tender with `human_verdict`, `actual_category`, and notes. Use `?` when the title is too ambiguous to judge.

In [ ]:
annotations = p4.load_annotations()
eval_rows = p4.evaluation_rows(records, sample_rows, annotations)

valid_annotations = [row for row in eval_rows if p4.is_annotated(row)]
print("Valid annotations:", len(valid_annotations))
print("Uncertain labels:", sum(row.get("human_verdict") == "?" for row in eval_rows))
print("Verdicts:")
from collections import Counter
print(Counter(row.get("human_verdict", "") for row in eval_rows))

## Tasks 3 and 4 - Metrics and Baseline

Soft metrics exclude `?` rows. Strict accuracy counts `?` as wrong. Headline accuracy is computed on the proportional subset; all 125 rows are used for diagnostic per-class behavior.

In [ ]:
soft_all = p4.soft_rows(eval_rows)
soft_rep = p4.soft_rows(eval_rows, representative_only=True)
strict_all = p4.strict_rows(eval_rows)
strict_rep = p4.strict_rows(eval_rows, representative_only=True)

metrics = p4.metrics_by_class(soft_all)
p4.print_metrics(metrics)

dataset_baseline = p4.dataset_baseline(records)
rep_accuracy = p4.accuracy(soft_rep)
rep_strict_accuracy = p4.accuracy(strict_rep, uncertain_is_wrong=True)

print(f"\nAll-row diagnostic accuracy (soft n={len(soft_all)}): {p4.accuracy(soft_all):.3f}")
print(f"All-row strict accuracy (? = wrong, n={len(strict_all)}): {p4.accuracy(strict_all, uncertain_is_wrong=True):.3f}")
print(f"Representative accuracy (soft n={len(soft_rep)}): {rep_accuracy:.3f}")
print(f"Representative strict accuracy (? = wrong, n={len(strict_rep)}): {rep_strict_accuracy:.3f}")
print(f"Dataset baseline, always Works: {dataset_baseline:.3f}")
print(f"Margin over dataset baseline: {rep_accuracy - dataset_baseline:+.3f}")

## Task 5 - Confusion Matrix

In [ ]:
matrix = p4.confusion_matrix_counts(soft_all)
p4.print_confusion_matrix(matrix)

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(len(p4.LABELS)), p4.LABELS)
    ax.set_yticks(range(len(p4.LABELS)), p4.LABELS)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix")
    for y, row in enumerate(matrix):
        for x, value in enumerate(row):
            ax.text(x, y, value, ha="center", va="center", color="black")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
except ModuleNotFoundError:
    print("matplotlib is not installed; the text matrix above is the verified output.")

## Task 6 - Error Analysis

In [ ]:
failures = p4.failure_counts(soft_all)
print("Failure type counts:")
for failure, count in failures.most_common():
    print(f"  {failure}: {count}")

print("\nWorst 5 high-confidence wrong predictions:")
for row in p4.worst_examples(soft_all):
    print(f"\n{row['tender_id']}")
    print(f"Title: {row['title']}")
    print(f"Predicted: {row['predicted_category']} ({float(row['category_confidence']):.3f})")
    print(f"Actual: {row['actual_category']}")
    print(f"Failure: {p4.classify_failure(row)}")

## Task 7 - Ground-Truth Cross-Check

In [ ]:
for check in p4.ground_truth_cross_check(records):
    print(
        f"{check['tender_id']}: "
        f"phase2={check['phase2_manual_bart']}, "
        f"phase3={check['phase3_pipeline_bart']}, "
        f"ground_truth={check['ground_truth']}, "
        f"consistent={check['consistent']}"
    )

## Optional Stability Re-Check

Run this cell only when `torch`, `transformers`, and `models/bart-large-mnli` are present. It selects a low-confidence correct row; if none are below 0.3, it selects the lowest-confidence correct row.

In [ ]:
# p4.run_stability_check(eval_rows, repeats=100, threshold=0.3)

## README Metrics Table

In [ ]:
print("| Metric       | Goods     | Services  | Works     | Overall   |")
print("|-------------|-----------|-----------|-----------|-----------|")
print(f"| Precision   | {metrics['Goods'].precision:.4f}    | {metrics['Services'].precision:.4f}    | {metrics['Works'].precision:.4f}    | --        |")
print(f"| Recall      | {metrics['Goods'].recall:.4f}    | {metrics['Services'].recall:.4f}    | {metrics['Works'].recall:.4f}    | --        |")
print(f"| F1          | {metrics['Goods'].f1:.4f}    | {metrics['Services'].f1:.4f}    | {metrics['Works'].f1:.4f}    | --        |")
print(f"| Accuracy    | --         | --         | --         | {rep_accuracy:.4f}    |")
print(f"| Baseline*   | --         | --         | --         | {dataset_baseline:.4f}    |")
print("\n*Baseline = always predict Works, the majority class in the enriched dataset")